# Answer 


What are the top 10 pickup zones?

What are the top 10 dropoff zones?

What is the average fare?

What is the average trip distance?

How many trips have total_amount <= 0?

How many trips have trip_distance = 0?

Which payment type is most common?

What percentage of trips include a tip?


In [138]:
import duckdb

con = duckdb.connect("taxi_project")

In [139]:
con.sql("""
SHOW TABLES;
""")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ taxi_clean      │
│ yellow_taxi_raw │
│ zone_lookup     │
└─────────────────┘

In [140]:
con.sql("CREATE TABLE zone_lookup AS SELECT * from read_csv('../data/nyc_taxi/lookup/taxi_zone_lookup.csv')")

CatalogException: Catalog Error: Table with name "zone_lookup" already exists!

In [ ]:
# How many trips are in the dataset?


In [ ]:
# What is the date range?


In [ ]:
# What are the top 10 pickup zones?


In [ ]:
# What is the average fare?


In [ ]:
# What is the average trip distance?


In [ ]:
# How many trips have total_amount <= 0?

In [ ]:
# How many trips have trip_distance = 0?

In [ ]:
# What percentage of trips include a tip?

In [ ]:
con.sql("SELECT MAX(fare_amount) FROM taxi_clean LIMIT 1;")

┌──────────────────┐
│ max(fare_amount) │
│      double      │
├──────────────────┤
│        863372.12 │
└──────────────────┘

In [ ]:
con.sql("SELECT MIN(fare_amount) FROM taxi_clean LIMIT 1;")

┌──────────────────┐
│ min(fare_amount) │
│      double      │
├──────────────────┤
│              0.0 │
└──────────────────┘

In [ ]:
con.sql("SELECT * FROM taxi_clean LIMIT 5;")

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-01 00:18:38 │ 2025-01-01 00:26:59 │               1 │           1.6 │            229 │            237 │        10.0 │        3.0 │
│ 2025-01-01 00:32:40 │ 2025-01-01 00:35:13 │               1 │           0.5 │            236 │            237 │         5.1 │       2.02 │
│ 2025-01-01 00:44:04 │ 2025-01-01 00:46:01 │               1 │           0.6 │            141 │            141 │         5.1 │        2.0 │
│ 2025-01-01 

In [ ]:
con.sql("SELECT AVG(tip_amount) from taxi_clean;")

┌────────────────────┐
│  avg(tip_amount)   │
│       double       │
├────────────────────┤
│ 3.5558984188647567 │
└────────────────────┘

# what zone pays the highest tips  

In [ ]:
con.sql("SELECT * FROM zone_lookup")

┌────────────┬───────────────┬───────────────────────────┬──────────────┐
│ LocationID │    Borough    │           Zone            │ service_zone │
│   int64    │    varchar    │          varchar          │   varchar    │
├────────────┼───────────────┼───────────────────────────┼──────────────┤
│          1 │ EWR           │ Newark Airport            │ EWR          │
│          2 │ Queens        │ Jamaica Bay               │ Boro Zone    │
│          3 │ Bronx         │ Allerton/Pelham Gardens   │ Boro Zone    │
│          4 │ Manhattan     │ Alphabet City             │ Yellow Zone  │
│          5 │ Staten Island │ Arden Heights             │ Boro Zone    │
│          6 │ Staten Island │ Arrochar/Fort Wadsworth   │ Boro Zone    │
│          7 │ Queens        │ Astoria                   │ Boro Zone    │
│          8 │ Queens        │ Astoria Park              │ Boro Zone    │
│          9 │ Queens        │ Auburndale                │ Boro Zone    │
│         10 │ Queens        │ Baisley

In [ ]:
con.sql("DELETE FROM zone_lookup WHERE Borough = 'Unknown' ")

In [ ]:
con.sql("SELECT DISTINCT(Borough), COUNT(*) as Count FROM zone_lookup GROUP BY Borough ORDER BY Count DESC;")

┌───────────────┬───────┐
│    Borough    │ Count │
│    varchar    │ int64 │
├───────────────┼───────┤
│ Manhattan     │    69 │
│ Queens        │    69 │
│ Brooklyn      │    61 │
│ Bronx         │    43 │
│ Staten Island │    20 │
│ EWR           │     1 │
└───────────────┴───────┘

In [ ]:
con.sql(
    """
    ALTER TABLE zone_lookup
    ADD nyc_class VARCHAR(50);
    UPDATE zone_lookup
    SET nyc_class = CASE
                        WHEN Borough IN ('Manhattn', 'Queens') THEN 'Rich NYC'
                        ELSE 'Poor NYC'
                    END;
    """
)

In [ ]:
con.sql("""
SELECT * from zone_lookup;
""")

┌────────────┬───────────────┬───────────────────────────┬──────────────┬───────────┐
│ LocationID │    Borough    │           Zone            │ service_zone │ nyc_class │
│   int64    │    varchar    │          varchar          │   varchar    │  varchar  │
├────────────┼───────────────┼───────────────────────────┼──────────────┼───────────┤
│          1 │ EWR           │ Newark Airport            │ EWR          │ Poor NYC  │
│          2 │ Queens        │ Jamaica Bay               │ Boro Zone    │ Rich NYC  │
│          3 │ Bronx         │ Allerton/Pelham Gardens   │ Boro Zone    │ Poor NYC  │
│          4 │ Manhattan     │ Alphabet City             │ Yellow Zone  │ Poor NYC  │
│          5 │ Staten Island │ Arden Heights             │ Boro Zone    │ Poor NYC  │
│          6 │ Staten Island │ Arrochar/Fort Wadsworth   │ Boro Zone    │ Poor NYC  │
│          7 │ Queens        │ Astoria                   │ Boro Zone    │ Rich NYC  │
│          8 │ Queens        │ Astoria Park           

In [ ]:
con.sql("ALTER TABLE zone_lookup DROP COLUMN nyc_class;")

In [ ]:
con.sql("""
    select * from zone_lookup where Borough = 'Manhattan';
""")

┌────────────┬───────────┬───────────────────────────┬──────────────┐
│ LocationID │  Borough  │           Zone            │ service_zone │
│   int64    │  varchar  │          varchar          │   varchar    │
├────────────┼───────────┼───────────────────────────┼──────────────┤
│          4 │ Manhattan │ Alphabet City             │ Yellow Zone  │
│         12 │ Manhattan │ Battery Park              │ Yellow Zone  │
│         13 │ Manhattan │ Battery Park City         │ Yellow Zone  │
│         24 │ Manhattan │ Bloomingdale              │ Yellow Zone  │
│         41 │ Manhattan │ Central Harlem            │ Boro Zone    │
│         42 │ Manhattan │ Central Harlem North      │ Boro Zone    │
│         43 │ Manhattan │ Central Park              │ Yellow Zone  │
│         45 │ Manhattan │ Chinatown                 │ Yellow Zone  │
│         48 │ Manhattan │ Clinton East              │ Yellow Zone  │
│         50 │ Manhattan │ Clinton West              │ Yellow Zone  │
│          · │     ·

In [ ]:
con.sql("""
    SELECT t.tip_amount, pu.Borough AS Pickup_borough, do_z.Borough AS dropoff_borough
    FROM taxi_clean AS t 
    INNER JOIN zone_lookup AS pu 
        ON t.pu_location_id = pu.LocationID
    INNER JOIN zone_lookup AS do_z
        ON t.do_location_id = do_z.LocationID
    WHERE t.tip_amount > 50
    ORDER BY t.tip_amount
        DESC
    LIMIT 10;
""")

┌────────────┬────────────────┬─────────────────┐
│ tip_amount │ Pickup_borough │ dropoff_borough │
│   double   │    varchar     │     varchar     │
├────────────┼────────────────┼─────────────────┤
│      440.0 │ Manhattan      │ Manhattan       │
│      400.0 │ Manhattan      │ Queens          │
│      333.3 │ Manhattan      │ Manhattan       │
│      303.0 │ Manhattan      │ Manhattan       │
│      285.0 │ Manhattan      │ Queens          │
│     284.96 │ Manhattan      │ Brooklyn        │
│      261.0 │ Manhattan      │ Queens          │
│     225.05 │ Manhattan      │ Manhattan       │
│      225.0 │ Manhattan      │ Manhattan       │
│      220.0 │ Manhattan      │ Manhattan       │
└────────────┴────────────────┴─────────────────┘
  10 rows                             3 columns

In [ ]:
con.sql("""
    SELECT COUNT(*) FROM taxi_clean;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      8554423 │
└──────────────┘

In [141]:
con.sql("""
    SELECT COUNT(*) FROM zone_lookup;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          263 │
└──────────────┘

In [142]:
con.close()